![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx to run AI service and switch between LLMs by updating the deployment

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook provides a detailed demonstration of the steps and code required to showcase support for watsonx.ai AI service.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal is to demonstrate how an AI service deployment using one LLM can be switched to another LLM of choice with zero downtime. It also highlights how an AI service asset can create a new revision and update the deployment accordingly.

## Table of Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Create AI service](#Create-AI-service)
3. [Testing AI service's function locally](#Testing-AI-service's-function-locally)
4. [Deploy AI service](#Deploy-AI-service)
5. [Example of executing an AI service](#Example-of-executing-an-AI-service)
6. [Create AI service revision](#Create-AI-service-revision)
7. [Update deployment with new revision](#Update-deployment-with-new-revision)
8. [Example of executing an AI service with updated deployment](#Example-of-executing-an-AI-service-with-updated-deployment)
9. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies

In [1]:
%pip install -U "ibm_watsonx_ai>=1.3.33" | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have a space, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials, space_id=space_id)

#### Specify model

This notebook uses chat models `meta-llama/llama-3-3-70b-instruct` and `openai/gpt-oss-20b`, which have to be available on your IBM Cloud Pak® for Data environment for this notebook to run successfully. If these models are not available on your IBM Cloud Pak® for Data environment, you can specify any other available chat models.

You can list available chat models by running the cell below.

In [6]:
if len(api_client.foundation_models.ChatModels):
    print(*api_client.foundation_models.ChatModels, sep="\n")
else:
    print(
        "Chat models are missing in this environment. Install chat models to proceed."
    )

meta-llama/llama-3-3-70b-instruct
openai/gpt-oss-20b


<a id="Create-AI-service"></a>
## Create AI service

Prepare function which will be deployed using AI service.

The below example uses `meta-llama/llama-3-3-70b-instruct` as its `model_id`

In [7]:
def deployable_ai_service(
    context, model_id="meta-llama/llama-3-3-70b-instruct", url=url
):
    from ibm_watsonx_ai import APIClient, Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference
    from ibm_watsonx_ai.foundation_models.schema import TextChatParameters

    parameters = TextChatParameters(
        temperature=1,
        max_completion_tokens=1000,
        top_p=1,
    )

    # token, and space_id are available from context object
    api_client = APIClient(
        credentials=Credentials(
            url=url,
            token=context.generate_token(),
            instance_id="openshift",
            version="5.4",
        ),
        space_id=context.get_space_id(),
    )

    model = ModelInference(
        model_id=model_id,
        api_client=api_client,
        params=parameters,
    )

    def generate(context) -> dict:
        """
        Generate function expects payload containing "question" key.

        Request json example:
        {
            "question": "<your question>"
        }

        Response body will provide answer under key: "answer".
        """
        # set the token for the inference user
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]
        chat_response = model.chat([{"role": "user", "content": question}])

        return {
            "body": {
                "answer": chat_response["choices"][0]["message"]["content"],
                "model_id": model.model_id,
            }
        }

    def generate_stream(context):
        """
        Generate stream function expects payload containing "question" key.

        Request json example:
        {
            "question": "<your question>"
        }

        The answer is returned as stream.
        """
        # set the token for the inference user
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]

        yield from (
            {"chunk": chunk["choices"][0]["delta"].get("content", "")}
            for chunk in model.chat_stream([{"role": "user", "content": question}])
            if chunk["choices"]
        )

    return generate, generate_stream

<a id="Testing-AI-service's-function-locally"></a>
## Testing AI service's function locally

You can test AI service's function locally. Initialize `RuntimeContext` firstly.

In [8]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(
    api_client=api_client, request_payload_json={"question": "What is inertia?"}
)

generate, generate_stream = deployable_ai_service(context)

Execute the `generate` function locally.

In [9]:
response = generate(context)
print(response["body"]["model_id"])
print(response["body"]["answer"])

meta-llama/llama-3-3-70b-instruct
Inertia is the tendency of an object to resist changes in its state of motion. According to Newton's first law of motion, an object at rest will remain at rest, and an object in motion will continue to move with a constant velocity, unless acted upon by an external force. This means that an object will maintain its state of motion unless a force is applied to it, and the more massive the object, the more inertia it has.


Execute the `generate_stream` function locally.

In [10]:
for data in generate_stream(context):
    print(data["chunk"], end="", flush=True)

Inertia is the resistance of an object to changes in its state of motion or rest. It is a fundamental concept in physics that describes the tendency of an object to maintain its current velocity and direction, unless acted upon by an external force. In other words, an object at rest will remain at rest, and an object in motion will continue to move with a constant velocity, unless a force is applied to it.

This concept was first described by Sir Isaac Newton in his First Law of Motion, which states that an object will remain in its state of motion unless acted upon by an external force. Inertia is a key concept in understanding many physical phenomena, including the motion of objects, the behavior of fluids, and the response of materials to stress and strain.

In simple terms, inertia is what keeps you in your seat when a car suddenly accelerates or decelerates, and it's what makes it difficult to change the direction of a moving object. It's an important concept to understand in many

<a id="Deploy-AI-service"></a>
## Deploy AI service

Store AI service which uses `meta-llama/llama-3-3-70b-instruct`

In [11]:
meta_props = {
    api_client.repository.AIServiceMetaNames.NAME: "AI service Q&A meta-llama/llama-3-3-70b-instruct",
    api_client.repository.AIServiceMetaNames.DESCRIPTION: "Test for patching model_id",
    api_client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: api_client.software_specifications.get_id_by_name(
        "runtime-25.1-py3.12"
    ),
}

stored_ai_service_details = api_client.repository.store_ai_service(
    deployable_ai_service, meta_props
)

ai_service_id = api_client.repository.get_ai_service_id(stored_ai_service_details)
print("The AI service asset id:", ai_service_id)

The AI service asset id: 019e1c43-4c82-7149-a40e-8c862a5ce17b


Create online deployment of AI service and obtain the `deployment_id`

In [12]:
deployment_details = api_client.deployments.create(
    artifact_id=ai_service_id,
    meta_props={
        api_client.deployments.ConfigurationMetaNames.NAME: "ai-service Q&A test",
        api_client.deployments.ConfigurationMetaNames.ONLINE: {},
        api_client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {
            "id": api_client.hardware_specifications.get_id_by_name("XXS")
        },
    },
)
dep_id = api_client.deployments.get_id(deployment_details)
dep_id



######################################################################################

Synchronous deployment creation for id: '019e1c43-4c82-7149-a40e-8c862a5ce17b' started

######################################################################################


initializing


Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
.........
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='019e1c43-758f-7570-b116-b9a1dbaa1db8'
-----------------------------------------------------------------------------------------------




'019e1c43-758f-7570-b116-b9a1dbaa1db8'

<a id="Example-of-executing-an-AI-service"></a>
## Example of executing an AI service

Execute `generate` method.

In [13]:
ai_service_payload = {"question": "What is inertia?"}
result = api_client.deployments.run_ai_service(
    deployment_id=dep_id, ai_service_payload=ai_service_payload
)
print(result["model_id"])
print(result["answer"])

meta-llama/llama-3-3-70b-instruct
Inertia is the tendency of an object to resist changes in its state of motion. This means that an object at rest will remain at rest, and an object in motion will continue to move with a constant velocity, unless acted upon by an external force. The concept of inertia is a fundamental principle in physics and is described by Newton's First Law of Motion.


Execute `generate_stream` method.

In [14]:
import json

ai_service_payload = {"question": "What is inertia?"}
for data in api_client.deployments.run_ai_service_stream(
    deployment_id=dep_id, ai_service_payload=ai_service_payload
):
    print(json.loads(data)["chunk"], end="", flush=True)

Inertia is the tendency of an object to resist changes in its state of motion. According to Newton's first law of motion, an object at rest will remain at rest, and an object in motion will continue to move with a constant velocity, unless acted upon by an external force. This means that an object will maintain its state of motion unless a force is applied to it, and the more massive the object, the more inertia it has.

<a id="Create-AI-service-revision"></a>
## Create AI service revision

We want to update the LLM the AI service uses from `meta-llama/llama-3-3-70b-instruct` to `openai/gpt-oss-20b`.  
For this we will update and create revision AI service asset followed by patching the deployment with the new revision.

_In this notebook we have the AI service function already available to us. However, in case it is not available it can be downloaded as shown below._

Download the existing AI service asset as a GZIP file. In order to edit it, decompression is needed.

In [15]:
api_client.repository.download(ai_service_id, "my_ai_svc.py.gz")

Successfully saved AI service content to file: 'my_ai_svc.py.gz'


In [16]:
!gunzip -fk my_ai_svc.py.gz

You can use notebook magic command `%load my_ai_svc.py` to load the contents and make the necessary changes to the content by replacing `model_id` with `openai/gpt-oss-20b`.

In [17]:
# %load my_ai_svc.py


def deployable_ai_service(context, model_id="openai/gpt-oss-20b", url=url):
    from ibm_watsonx_ai import APIClient, Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference
    from ibm_watsonx_ai.foundation_models.schema import TextChatParameters

    parameters = TextChatParameters(
        temperature=1,
        max_completion_tokens=1000,
        top_p=1,
    )

    # token, and space_id are available from context object
    api_client = APIClient(
        credentials=Credentials(
            url=url,
            token=context.generate_token(),
            instance_id="openshift",
            version="5.4",
        ),
        space_id=context.get_space_id(),
    )

    model = ModelInference(
        model_id=model_id,
        api_client=api_client,
        params=parameters,
    )

    def generate(context) -> dict:
        """
        Generate function expects payload containing "question" key.

        Request json example:
        {
            "question": "<your question>"
        }

        Response body will provide answer under key: "answer".
        """
        # set the token for the inference user
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]
        chat_response = model.chat([{"role": "user", "content": question}])

        return {
            "body": {
                "answer": chat_response["choices"][0]["message"]["content"],
                "model_id": model.model_id,
            }
        }

    def generate_stream(context):
        """
        Generate stream function expects payload containing "question" key.

        Request json example:
        {
            "question": "<your question>"
        }

        The answer is returned as stream.
        """
        # set the token for the inference user
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]

        yield from (
            {"chunk": chunk["choices"][0]["delta"].get("content", "")}
            for chunk in model.chat_stream([{"role": "user", "content": question}])
            if chunk["choices"]
        )

    return generate, generate_stream

Optional step: create revision for the existing version for safe keeping.


In [18]:
response = api_client.repository.create_ai_service_revision(ai_service_id)
print(json.dumps(response, indent=2))

Update the AI service asset with the new content

In [19]:
print("Updating content for AI service:", ai_service_id)
ai_service_details = api_client.repository.update_ai_service(
    ai_service_id,
    changes={
        api_client.repository.AIServiceMetaNames.NAME: "AI service Q&A openai/gpt-oss-20b"
    },
    update_ai_service=deployable_ai_service,
)

print(json.dumps(ai_service_details, indent=2))

Create revision for the new content

In [20]:
ai_service_details_for_patch = api_client.repository.create_ai_service_revision(
    ai_service_id
)
print(json.dumps(ai_service_details_for_patch, indent=2))

In [21]:
rev = ai_service_details_for_patch["metadata"]["rev"]
print("The required revision:", rev)

The required revision: 2


<a id="Update-deployment-with-new-revision"></a>
## Update deployment with new revision

In [22]:
updated_deployment_details = api_client.deployments.update(
    deployment_id=dep_id,
    changes={
        api_client.deployments.ConfigurationMetaNames.ASSET: {
            "id": ai_service_id,
            "rev": rev,
        }
    },
)

Since ASSET is patched, deployment need to be restarted.


########################################################################

Deployment update for id: '019e1c43-758f-7570-b116-b9a1dbaa1db8' started

########################################################################


updating........
ready


---------------------------------------------------------------------------------------------
Successfully finished deployment update, deployment_id='019e1c43-758f-7570-b116-b9a1dbaa1db8'
---------------------------------------------------------------------------------------------




The deployment now be reflects the new asset revision

In [23]:
updated_deployment_details["entity"]["asset"]

{'id': '019e1c43-4c82-7149-a40e-8c862a5ce17b', 'rev': '2'}

<a id="Example-of-executing-an-AI-service-with-updated-deployment"></a>
## Example of executing an AI service with updated deployment

Execute `generate` method.

In [24]:
ai_service_payload = {"question": "What is inertia?"}
result = api_client.deployments.run_ai_service(
    deployment_id=dep_id, ai_service_payload=ai_service_payload
)
print(result["model_id"])
print(result["answer"])

openai/gpt-oss-20b
### Inertia – the “stickiness” of motion

**Inertia** is the property of matter that makes it resist changes in its state of motion. In everyday language you might say an object “has a lot of inertia” when it is hard to set moving or hard to bring to a stop.

---

## 1. Newton’s First Law (the Law of Inertia)

> *An object at rest stays at rest, and an object in motion continues in uniform motion in a straight line unless acted upon by an external force.*

The word “inertia” comes from this law. It tells us that:
- A stationary object will not start moving unless a force pushes or pulls on it.
- A moving object will keep going in a straight line with constant speed unless a force changes its motion (slows it down, speeds it up, or turns it).

---

## 2. How it depends on mass

- **Mass** is the quantitative measure of inertia.  
  - The more mass an object has, the more inertia it possesses.  
  - A truck needs a lot more force to change speed than a bicycle because 

Execute `generate_stream` method.

In [25]:
ai_service_payload = {"question": "What is inertia?"}
for data in api_client.deployments.run_ai_service_stream(
    deployment_id=dep_id, ai_service_payload=ai_service_payload
):
    print(json.loads(data)["chunk"], end="", flush=True)

**Inertia** is a property of matter that resists changes to its motion. In everyday terms it’s why a moving car stalls when the driver’s foot leaves the gas pedal, or why a spinning top slows down when the air friction tries to stop it.

There are two common ways to talk about inertia:

| Concept | What it measures | Physical meaning |
|---------|------------------|------------------|
| **Linear inertia** | Mass (kg, lb⋅s²/m) | The amount of “push‑back” a body offers against forces that change its translational speed or direction. |
| **Rotational inertia** | Moment of inertia (kg·m², slug·ft²) | The resistance of a body to changes in its angular velocity, depending on how its mass is distributed from the axis of rotation. |


---

### Newton’s First Law  
> *Everything stays at rest or in uniform motion unless acted upon by a net external force.*

Inertia is the underlying cause of that “tendency” to keep moving (or stay still). With a larger amount of inertia (larger mass or moment o

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use the `ibm_watsonx_ai` SDK to create an AI service asset, update its deployment by creating a new revision, and switch the deployment to a different LLM of choice.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Ginbiaksang Naulak (Former)**, Senior Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.